# Train Forecasting Model

## Imports

In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error

import xgboost as xgb

import matplotlib.pyplot as plt
import seaborn as sns

import joblib

## Loading Data

In [2]:
PROCESSED_DATA_DIR= '../data/processed'
MODELS_DIR= '../models'

In [3]:
data_path= os.path.join(PROCESSED_DATA_DIR, 'forecasting_training_data.csv')

In [4]:
df= pd.read_csv(filepath_or_buffer= data_path)

In [5]:
df.head()

,sale_date,category,units_sold,daily_revenue
0,2016-09-15,health_beauty,3,134.97
1,2016-10-03,fashion_shoes,1,29.99
2,2016-10-03,furniture_decor,2,194.80
3,2016-10-03,sports_leisure,2,58.39
4,2016-10-03,toys,1,128.90


In [6]:
df.describe()

,units_sold,daily_revenue
count,18311.000000,18311.000000
mean,5.932936,712.442297
std,7.285815,969.197817
min,1.000000,3.850000
25%,1.000000,119.730000
50%,3.000000,349.900000
75%,8.000000,923.335000
max,192.000000,17667.020000


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18311 entries, 0 to 18310
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   sale_date      18311 non-null  object 
 1   category       18311 non-null  object 
 2   units_sold     18311 non-null  int64  
 3   daily_revenue  18311 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 572.3+ KB


In [8]:
df.shape

(18311, 4)

## Feature Engineering

In [9]:
# Converting sale_date to DateTime:
df['sale_date'] = pd.to_datetime(df['sale_date'])

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18311 entries, 0 to 18310
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   sale_date      18311 non-null  datetime64[ns]
 1   category       18311 non-null  object        
 2   units_sold     18311 non-null  int64         
 3   daily_revenue  18311 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 572.3+ KB


In [11]:
# Re-Sampling to Weekly Buckets by Category:
weekly_df= df.groupby('category').resample('W', on='sale_date').agg({
    'daily_revenue':'sum',
    'units_sold': 'sum'
}).reset_index()

In [12]:
weekly_df.head()

,category,sale_date,daily_revenue,units_sold
0,agro_industry_and_commerce,2017-01-29,43.98,2
1,agro_industry_and_commerce,2017-02-05,43.98,2
2,agro_industry_and_commerce,2017-02-12,114.89,2
3,agro_industry_and_commerce,2017-02-19,65.97,3
4,agro_industry_and_commerce,2017-02-26,21.99,1


In [13]:
# Renaming Columns:
weekly_df.rename(columns={
    'sale_date': 'week_ending_date',
    'daily_revenue': 'weekly_revenue',
    'units_sold': 'weekly_units'
}, inplace= True)

In [14]:
weekly_df.head()

,category,week_ending_date,weekly_revenue,weekly_units
0,agro_industry_and_commerce,2017-01-29,43.98,2
1,agro_industry_and_commerce,2017-02-05,43.98,2
2,agro_industry_and_commerce,2017-02-12,114.89,2
3,agro_industry_and_commerce,2017-02-19,65.97,3
4,agro_industry_and_commerce,2017-02-26,21.99,1


In [15]:
# Extracting Calendar Features:
weekly_df['year']= weekly_df['week_ending_date'].dt.isocalendar().year
weekly_df['month']= weekly_df['week_ending_date'].dt.month
weekly_df['week_of_year']= weekly_df['week_ending_date'].dt.isocalendar().week

In [16]:
weekly_df.head()

,category,week_ending_date,weekly_revenue,weekly_units,year,month,week_of_year
0,agro_industry_and_commerce,2017-01-29,43.98,2,2017,1,4
1,agro_industry_and_commerce,2017-02-05,43.98,2,2017,2,5
2,agro_industry_and_commerce,2017-02-12,114.89,2,2017,2,6
3,agro_industry_and_commerce,2017-02-19,65.97,3,2017,2,7
4,agro_industry_and_commerce,2017-02-26,21.99,1,2017,2,8


In [17]:
# Adding Auto-Regressive / LAG Features:
# 1. Last week:
weekly_df['lag_1_revenue']= weekly_df.groupby('category')['weekly_revenue'].shift(1)
weekly_df['lag_1_units']= weekly_df.groupby('category')['weekly_units'].shift(1)

In [18]:
weekly_df.head()

,category,week_ending_date,weekly_revenue,weekly_units,year,month,week_of_year,lag_1_revenue,lag_1_units
0,agro_industry_and_commerce,2017-01-29,43.98,2,2017,1,4,NaN,NaN
1,agro_industry_and_commerce,2017-02-05,43.98,2,2017,2,5,43.98,2.0
2,agro_industry_and_commerce,2017-02-12,114.89,2,2017,2,6,43.98,2.0
3,agro_industry_and_commerce,2017-02-19,65.97,3,2017,2,7,114.89,2.0
4,agro_industry_and_commerce,2017-02-26,21.99,1,2017,2,8,65.97,3.0


In [19]:
# EWMA Prioritizing last 4 Weeks:
weekly_df['ewma_4_revenue']= weekly_df.groupby('category')['weekly_revenue'].transform(
    lambda x: x.ewm(span= 4, adjust=False).mean().shift(1)
)

weekly_df['ewma_4_units']= weekly_df.groupby('category')['weekly_units'].transform(
    lambda x: x.ewm(span= 4, adjust=False).mean().shift(1)
)

In [20]:
weekly_df.head()

,category,week_ending_date,weekly_revenue,weekly_units,year,month,week_of_year,lag_1_revenue,lag_1_units,ewma_4_revenue,ewma_4_units
0,agro_industry_and_commerce,2017-01-29,43.98,2,2017,1,4,NaN,NaN,NaN,NaN
1,agro_industry_and_commerce,2017-02-05,43.98,2,2017,2,5,43.98,2.0,43.9800,2.0
2,agro_industry_and_commerce,2017-02-12,114.89,2,2017,2,6,43.98,2.0,43.9800,2.0
3,agro_industry_and_commerce,2017-02-19,65.97,3,2017,2,7,114.89,2.0,72.3440,2.0
4,agro_industry_and_commerce,2017-02-26,21.99,1,2017,2,8,65.97,3.0,69.7944,2.4


In [21]:
# Dropping Null Values Created by LAG Features:
weekly_df= weekly_df.dropna().reset_index(drop= True)
weekly_df.head()

,category,week_ending_date,weekly_revenue,weekly_units,year,month,week_of_year,lag_1_revenue,lag_1_units,ewma_4_revenue,ewma_4_units
0,agro_industry_and_commerce,2017-02-05,43.98,2,2017,2,5,43.98,2.0,43.98000,2.00
1,agro_industry_and_commerce,2017-02-12,114.89,2,2017,2,6,43.98,2.0,43.98000,2.00
2,agro_industry_and_commerce,2017-02-19,65.97,3,2017,2,7,114.89,2.0,72.34400,2.00
3,agro_industry_and_commerce,2017-02-26,21.99,1,2017,2,8,65.97,3.0,69.79440,2.40
4,agro_industry_and_commerce,2017-03-05,0.00,0,2017,3,9,21.99,1.0,50.67264,1.84


In [22]:
weekly_df.shape

(5919, 11)

## Train Test Split

In [23]:
# Cutoff Date for Last 8 weeks:
cutoff_date= weekly_df['week_ending_date'].max() - pd.Timedelta(weeks= 8)

# Training Data:
train_df= weekly_df[weekly_df['week_ending_date'] < cutoff_date]
test_df= weekly_df[weekly_df['week_ending_date'] >= cutoff_date]

In [24]:
print(train_df.shape)
print(test_df.shape)

(5387, 11)
(532, 11)


In [ ]:
# 